# Does `aoutools` score the *real* All of Us VDS correctly?

Run this on the **All of Us Researcher Workbench**, on a Hail Genomic
Analysis environment. It is the real-data counterpart of
`tests/integration/`, and it exists because that suite has one weakness it
cannot fix by itself: it runs against a **hand-built mock VDS**.

Every scenario in `tests/integration/conftest.py` -- the hom-ref sample with
no entry, the no-call, the multi-allelic site, the non-minimally-represented
variant -- is a *claim about how the real VDS is shaped*. If a claim is
wrong, the mock suite stays green and the scores are wrong anyway.

This notebook takes each of those fixtures, **finds a real variant of the
same shape in the real VDS**, and checks the library's score against a
ground truth computed independently of `aoutools`.

## The oracle

Expected scores are copy numbers counted from `hl.vds.to_dense_mt`, where
homozygous-reference samples have been reconstituted from the reference
blocks and are therefore actually visible. Every weight is `1.0`, so a
sample's expected PRS **is** its count of effect-allele copies.

The oracle counts copies of a specific **allele index** in the row's original,
unsplit allele list. It never calls `min_rep`, never splits, and never calls
`aoutools`. That independence is the point: it is what caught **Finding 6**
(see `TODO.md`) while this notebook was being written.

One subtlety it has to get right: a **no-call** -- an entry whose genotype is
missing -- contributes **0**, not two copies of the reference. It is an
*unknown* genotype, not a hom-ref one. Conflating those two is the bug that
sank the old non-split scoring path.

## If a check fails

A red check means a real bug in the library, or a false assumption baked into
`tests/integration/`. Either way it is a finding. **Do not adjust the oracle
to make it pass.**

## Setup

`aoutools.init_hail()` handles the two things the Workbench needs: the
requester-pays billing project (from `$GOOGLE_PROJECT`), and the reference
genome, which is now set *after* `hl.init` rather than passed to it.

`aoutools.get_vds_path()` prefers `$WGS_VDS_PATH` and falls back to the
current All of Us release, with a warning, if that variable is not set --
which it is not on current Workbench images.

In [ ]:
# If aoutools isn't already installed in this environment:
# !pip install -q git+https://github.com/dokyoonkimlab/aoutools.git@dev

import logging
import time

import hail as hl
import pandas as pd

from aoutools import get_vds_path, init_hail
from aoutools.prs._calculator import _calculate_prs_chunk, _process_chunks
from aoutools.prs._calculator_batch import (
    _calculate_prs_chunk_batch,
    _prepare_batch_weights_data,
)
from aoutools.prs._calculator_utils import (
    _prepare_weights_for_chunking,
    _validate_and_prepare_weights_table,
)
from aoutools.prs._config import PRSConfig

# aoutools logs progress through the standard logging module at INFO but ships
# only a NullHandler, so nothing prints until a handler is attached. Attach one
# (`force=True` replaces the handler Jupyter puts on the root logger). Root
# stays at WARNING to mute Spark/py4j; only aoutools is raised to INFO, which
# surfaces the per-chunk "Processing chunk" progress line.
#
# This notebook scores one variant at a time, dozens of times, so it does NOT
# turn on PRSConfig(detailed_timings=True): that adds a per-phase breakdown to
# every one of those calls and would bury the checks. Use it for a single large
# score instead -- see validate_public_api_on_aou.ipynb.
logging.basicConfig(
    level=logging.WARNING,
    format="%(asctime)s  %(levelname)-7s  %(name)s  %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)
logging.getLogger("aoutools").setLevel(logging.INFO)

init_hail()

VDS_PATH = get_vds_path()
print("VDS:", VDS_PATH)

# Pair with the final cell to report end-to-end wall time.
NOTEBOOK_START = time.perf_counter()

`calculate_prs` and `calculate_pgs` insist on a `gs://` output path, so they
are exercised in `validate_public_api_on_aou.ipynb` instead. Here we score
through `_calculate_prs_chunk`, the same seam `tests/integration/` uses,
which hands back an in-memory table.

## Pick a cohort and a search window

Keep this small. The window only has to be wide enough to **contain** one
variant of each shape; once they are chosen, the VDS is cut down to just
those variants and everything after that is tiny.

If a category comes up empty below, widen `WINDOW`.

In [ ]:
WINDOW = "chr1:1000000-2000000"
N_SAMPLES = 200

vds_full = hl.vds.read_vds(VDS_PATH)

interval = hl.parse_locus_interval(WINDOW, reference_genome="GRCh38")
vds_win = hl.vds.filter_intervals(vds_full, [interval], keep=True)

sample_ids = vds_win.variant_data.cols().head(N_SAMPLES).s.collect()
samples_ht = hl.Table.parallelize(
    [{"s": s} for s in sample_ids], hl.tstruct(s=hl.tstr)
).key_by("s")
vds_win = hl.vds.filter_samples(vds_win, samples_ht)

print(f"{len(sample_ids)} samples, window {WINDOW}")

## Classify every variant in the window

Each row is sorted into the shape it corresponds to in
`tests/integration/conftest.py`. Classification is per **ALT allele**: each
`(REF, ALT_i)` pair is put into its minimal representation and compared with
the original.

* **suffix-trimmed, locus stable** -- `min_rep` shortened the alleles but the
  locus stayed put. This is the case the library *relies on*: VDS alleles
  `[AGGGC, A, GGGGC]` reduce to `A/G` at the same locus, which is how a GWAS
  names that SNP.
* **locus-shifting** -- `min_rep` *moved* the locus. `hl.vds.split_multi` will
  not relocate a row, so it can only raise. **We expect zero of these.** A
  previous measurement found 0 of 6,001,424 ALT alleles
  (`notebooks/measure_minrep_locus_shift.ipynb`). The assert below is a
  tripwire against a future change in All of Us variant representation.

The same pass also counts **no-call** entries per row, so we can find a real
one rather than assuming they exist.

In [ ]:
vd = vds_win.variant_data
vd = vd.annotate_rows(
    n_nocall=hl.agg.count_where(~hl.is_defined(vd.LGT)),
    n_entries=hl.agg.count(),
)
rows = vd.rows().select("n_nocall", "n_entries")

# min_rep each (REF, ALT_i) pair independently: one allele of a multi-allelic
# row can normalize cleanly while another does not.
rows = rows.annotate(
    minreps=hl.range(1, hl.len(rows.alleles)).map(
        lambda i: hl.min_rep(rows.locus, [rows.alleles[0], rows.alleles[i]])
    )
)
rows = rows.annotate(
    shifts=hl.any(rows.minreps.map(lambda m: m.locus != rows.locus)),
    suffix_trimmed=hl.any(
        rows.minreps.map(
            lambda m: (m.locus == rows.locus)
            & (hl.len(m.alleles[0]) < hl.len(rows.alleles[0]))
        )
    ),
    n_alleles=hl.len(rows.alleles),
)
rows = rows.checkpoint(hl.utils.new_temp_file(), overwrite=True)

counts = rows.aggregate(
    hl.struct(
        n_rows=hl.agg.count(),
        n_multiallelic=hl.agg.count_where(rows.n_alleles > 2),
        n_suffix_trimmed=hl.agg.count_where(rows.suffix_trimmed),
        n_with_nocall=hl.agg.count_where(rows.n_nocall > 0),
        n_shifting=hl.agg.count_where(rows.shifts),
    )
)
for k, v in counts.items():
    print(f"{k:20s} {v:>10,}")

assert counts.n_shifting == 0, (
    f"TRIPWIRE: {counts.n_shifting} locus-shifting ALT alleles found. "
    "hl.vds.split_multi cannot relocate a row, so these variants CANNOT be "
    "scored -- it raises. The All of Us variant representation has changed. "
    "Do NOT silence this with filter_changed_loci=True: that trades a loud "
    "failure for silently dropped variants. See TODO.md, Finding 5."
)
print("\nTripwire OK: no locus-shifting variants in this window.")

## Find one real variant per fixture

`tests/integration/conftest.py` hand-writes its variants. Here we go and find
the real thing.

In [ ]:
def key_of(v):
    """Identity of a variant: its locus and its full allele list."""
    return (str(v.locus), tuple(v.alleles))


def pick(expr, n=1):
    """Up to n rows of the window matching a boolean expression."""
    return rows.filter(expr & ~rows.shifts).head(n).collect()


def pick_distinct(expr, taken, n_scan=50):
    """First row matching `expr` whose variant is not already `taken`.

    The shape categories are NOT mutually exclusive: a real variant can be
    multi-allelic AND non-minimally represented at the same time -- the first
    such row this window offers is exactly that, `['ATCCCGGC', 'A',
    'CTCCCGGC', 'GTCCCGGC']`. Letting one row stand for two categories is
    harmless to the checks themselves, but it makes the selection shorter than
    it looks, so prefer a fresh row and only reuse one if the window has no
    alternative.
    """
    for v in rows.filter(expr & ~rows.shifts).head(n_scan).collect():
        if key_of(v) not in taken:
            return [v]
    return pick(expr)


is_snp = (
    (rows.n_alleles == 2)
    & (hl.len(rows.alleles[0]) == 1)
    & (hl.len(rows.alleles[1]) == 1)
)

FOUND = {}
taken = set()


def take(name, *exprs):
    """First expression that yields an unused variant wins; record it."""
    found = []
    for expr in exprs:
        found = pick_distinct(expr, taken)
        if found:
            break
    FOUND[name] = found
    taken.update(key_of(v) for v in found)


# conftest chr1:1000 / chr1:2000 -- a clean biallelic SNP.
take("biallelic", is_snp & (rows.n_nocall == 0))
# conftest chr1:2000's S4 -- an entry whose genotype is missing. No-calls are
# rare (27 rows in a 1Mb window), so take a SNP if there is one and any shape
# if not, rather than come up empty.
take("no_call", is_snp & (rows.n_nocall > 0), rows.n_nocall > 0)
# conftest chr1:5000 / chr1:7000 -- a multi-allelic site.
take("multiallelic", rows.n_alleles > 2)
# conftest chr1:6000 -- not minimally represented, locus does NOT move. Prefer
# a biallelic one so this is a genuinely different row from the multi-allelic
# check above.
take(
    "non_minimal",
    (rows.n_alleles == 2) & rows.suffix_trimmed,
    rows.suffix_trimmed,
)

for name, found in FOUND.items():
    if not found:
        print(f"{name:14s} NOT FOUND")
        continue
    v = found[0]
    print(
        f"{name:14s} {v.locus} {list(v.alleles)}"
        f"  ({v.n_entries} entries, {v.n_nocall} no-call)"
    )

assert FOUND["biallelic"], "no biallelic SNP found -- widen WINDOW"
assert FOUND["multiallelic"], "no multi-allelic site found -- widen WINDOW"
assert FOUND["non_minimal"], (
    "no suffix-trimmed variant found -- widen WINDOW. This is the "
    "normalization the library depends on, so it is worth finding one."
)
if not FOUND["no_call"]:
    print(
        "\nNo no-call entry in this window. The no-call check below will be "
        "skipped; widen WINDOW to include it."
    )

## Cut the VDS down to the chosen variants

From here the data is tiny, and nothing outside these variants can contribute
to a score. `reference_data` is untouched, so hom-ref samples remain
reconstitutable -- which is what the oracle needs.

In [ ]:
# The categories can land on the same row, so deduplicate. If they are not
# deduplicated, `filter_variants` still yields one VDS row per distinct variant
# while `len(SELECTED)` counts the duplicate, and every per-sample entry count
# below is then compared against a K that is too large.
SELECTED = []
seen = set()
for found in FOUND.values():
    if found and key_of(found[0]) not in seen:
        seen.add(key_of(found[0]))
        SELECTED.append(found[0])

variants_ht = hl.Table.parallelize(
    [{"locus": v.locus, "alleles": list(v.alleles)} for v in SELECTED],
    hl.tstruct(locus=hl.tlocus("GRCh38"), alleles=hl.tarray(hl.tstr)),
).key_by("locus", "alleles")

vds = hl.vds.filter_variants(vds_win, variants_ht, keep=True)
vds = vds.checkpoint(hl.utils.new_temp_file(extension="vds"), overwrite=True)

# Take K from the VDS, never from the list. The two disagreeing is the bug this
# assert exists to catch.
K = vds.variant_data.count_rows()
assert K == len(SELECTED), (
    f"the filtered VDS has {K} rows but {len(SELECTED)} distinct variants "
    "were selected -- they came from this VDS, so it should have all of them"
)

dense = hl.vds.to_dense_mt(vds)
dense = dense.annotate_entries(GT=hl.vds.lgt_to_gt(dense.LGT, dense.LA))
dense = dense.checkpoint(hl.utils.new_temp_file(), overwrite=True)

print(f"{K} distinct variants, {len(sample_ids)} samples")
for v in SELECTED:
    print(f"  {v.locus} {list(v.alleles)}")

## Check 1 -- a hom-ref sample is not an entry

The fact everything else rests on. In `variant_data`, a sample with no
non-reference call at a row is **absent**, and hail *filters absent entries
out of the entry stream* -- so aggregators never visit it and **no per-entry
default of any kind can reach it**.

This is why a REF-effect variant needs a row-level `2w` offset: it is the only
way to give a hom-ref sample a score at all.

In the dense matrix most of those samples reappear, because the reference
blocks have been expanded back out. The gap between the two counts is the
whole design.

*Most*, not all. A sample can be absent from `variant_data` for two different
reasons, and densifying tells them apart where the library cannot:

* it was **called homozygous reference** -- a reference block covers the
  locus, and densifying restores it;
* it was **not called at all** -- no variant entry and no reference block, so
  it stays missing even in the dense matrix.

The second is a genuinely *unknown* genotype, but from `variant_data` it looks
exactly like the first, so the library scores it as hom-ref. The count is
printed below and the checks report it separately; it is an imputation, not a
scoring error.

In [ ]:
sparse = (
    vds.variant_data.select_cols(n_sparse=hl.agg.count()).cols().to_pandas()
)
densec = dense.select_cols(n_dense=hl.agg.count()).cols().to_pandas()
entry_counts = sparse.merge(densec, on="s")

n_invisible = int((entry_counts.n_sparse == 0).sum())
n_no_data = int((K - entry_counts.n_dense).sum())

print(f"K = {K} variants, {len(entry_counts)} samples\n")
print("entries per sample in variant_data (sparse):")
print(entry_counts.n_sparse.value_counts().sort_index().to_string())
print("\nentries per sample in the dense matrix:")
print(entry_counts.n_dense.value_counts().sort_index().to_string())
print(f"\nsamples with ZERO entries in variant_data: {n_invisible}")

# Densifying reconstitutes hom-ref samples from the reference blocks, so it can
# only ADD entries -- never lose one that variant_data already has. If it did,
# the oracle and the library would not be looking at the same data.
assert (entry_counts.n_dense >= entry_counts.n_sparse).all(), (
    "densifying LOST an entry that variant_data has"
)

# The hom-ref offset is only exercised if some sample is absent from the entry
# stream at a scored variant. If none is, this window proves nothing about it.
assert n_invisible > 0, (
    "expected at least one sample to be hom-ref (and therefore absent from "
    "variant_data) at a selected variant -- if none is, this window cannot "
    "exercise the hom-ref offset at all"
)
print("\nOK: hom-ref samples are invisible to the entry stream.")

# NOT every sample need appear at every variant even once densified. A pair
# with no variant entry AND no reference block was never called there at all:
# its genotype is UNKNOWN, not hom-ref. The library cannot tell the two apart
# -- both are simply absent from variant_data -- so it scores such a sample as
# hom-ref. That is an imputation, and `check` below reports it separately
# rather than counting it as a scoring error.
print(f"\n(sample, variant) pairs with no data at all: {n_no_data}")
if n_no_data:
    print(
        "  These samples were not called at that locus. The library imputes "
        "hom-ref for them; the oracle calls them unknown. See the NOTE lines "
        "in the checks below."
    )

## The oracle, and the scoring harness

`minimal_representation` re-implements, in plain Python, the trimming a GWAS
does when it names a variant. It is used only to **write the weights file** --
to name the variant the way a GWAS would -- never to score anything.

`expected_copies` is the ground truth: it counts each sample's copies of one
chosen **allele index** in the row's original allele list. No `min_rep`, no
`split_multi`, no `aoutools`. A missing genotype is a no-call and counts 0. It
also reports, per sample, whether there is any data at that variant at all --
see check 1 for why that is not the same question.

`check` puts a single-variant weights file through the library and compares.

In [ ]:
def minimal_representation(ref, alt):
    """Names a variant the way a GWAS would. Pure Python, no hail.

    Trims the shared suffix, then the shared prefix, keeping at least one base
    of each allele. Returns (ref, alt, shift), where `shift` is how far the
    locus moves -- nonzero only if a shared prefix was trimmed.
    """
    while len(ref) > 1 and len(alt) > 1 and ref[-1] == alt[-1]:
        ref, alt = ref[:-1], alt[:-1]
    shift = 0
    while len(ref) > 1 and len(alt) > 1 and ref[0] == alt[0]:
        ref, alt = ref[1:], alt[1:]
        shift += 1
    return ref, alt, shift


# The two cases the library's behavior turns on.
assert minimal_representation("AGGGC", "GGGGC") == ("A", "G", 0)  # safe
assert minimal_representation("GG", "GT") == ("G", "T", 1)  # MOVES the locus


def expected_copies(variant, allele_index):
    """Ground truth: each sample's copies of `alleles[allele_index]`.

    Counted from the dense matrix, against the row's ORIGINAL allele list.
    Nothing here touches aoutools, min_rep, or split_multi.

    Returns (copies, has_data). `has_data` is False for a sample with no entry
    at all at this variant -- never called there, so its genotype is unknown
    rather than hom-ref, and `copies` is meaningless for it.
    """
    key = f"{variant.locus}|{','.join(variant.alleles)}"
    row_key = hl.str(dense.locus) + "|" + hl.delimit(dense.alleles, ",")
    idx = hl.literal({key: allele_index}).get(row_key, -1)
    at_row = idx >= 0

    copies = hl.if_else(
        hl.is_defined(dense.GT) & at_row,
        # A no-call is an UNKNOWN genotype and contributes 0 -- it is not two
        # copies of the reference.
        hl.array([dense.GT[0], dense.GT[1]]).filter(lambda a: a == idx).size(),
        0,
    )
    df = (
        dense.select_cols(
            n=hl.agg.sum(copies),
            # count_where only visits entries that EXIST, so this is 0 exactly
            # when the sample has no entry at this variant.
            has_data=hl.agg.count_where(at_row) > 0,
        )
        .cols()
        .to_pandas()
    )
    copies = dict(zip(df["s"], df["n"].astype(float), strict=True))
    has_data = dict(zip(df["s"], df["has_data"].astype(bool), strict=True))
    return copies, has_data


def library_score(weights_rows):
    """Scores the VDS with aoutools and returns ({sample: prs}, n_matched)."""
    raw = hl.Table.parallelize(
        weights_rows,
        hl.tstruct(
            chr=hl.tstr,
            pos=hl.tint32,
            effect_allele=hl.tstr,
            noneffect_allele=hl.tstr,
            weight=hl.tfloat64,
        ),
    )
    config = PRSConfig(include_n_matched=True)
    weights = _validate_and_prepare_weights_table(raw, config)
    df = _calculate_prs_chunk(weights, vds, config).to_pandas()
    prs = dict(zip(df["person_id"], df["prs"].astype(float), strict=True))
    return prs, int(df["n_matched"].iloc[0])


def weights_row(variant, alt_index, ref_is_effect):
    """A one-line GWAS summary naming (REF, ALT_i) at this variant."""
    ref, alt, shift = minimal_representation(
        variant.alleles[0], variant.alleles[alt_index]
    )
    assert shift == 0, "this variant's locus moves; it cannot be scored"
    effect, noneffect = (ref, alt) if ref_is_effect else (alt, ref)
    return {
        "chr": variant.locus.contig,
        "pos": variant.locus.position,
        "effect_allele": effect,
        "noneffect_allele": noneffect,
        "weight": 1.0,
    }


RESULTS = []


def check(name, variant, alt_index, ref_is_effect):
    """Scores one variant both ways and compares. Records a pass/fail."""
    row = weights_row(variant, alt_index, ref_is_effect)
    effect_index = 0 if ref_is_effect else alt_index

    expected, has_data = expected_copies(variant, effect_index)
    got, n_matched = library_score([row])

    # A real disagreement: the sample HAS a genotype here and the library got
    # its copy count wrong.
    mismatches = {
        s: (expected[s], got[s])
        for s in expected
        if has_data[s] and abs(expected[s] - got[s]) > 1e-9
    }
    # Not a disagreement: the sample has no genotype here at all. The oracle
    # can see that (it read the reference blocks); the library cannot, and
    # imputes hom-ref. Reported, not failed -- see check 1.
    n_imputed = sum(
        1
        for s in expected
        if not has_data[s] and abs(expected[s] - got[s]) > 1e-9
    )
    ok = n_matched == 1 and not mismatches
    RESULTS.append((name, ok, len(mismatches)))

    print(f"{'PASS' if ok else 'FAIL'}  {name}")
    print(
        f"      {variant.locus} {list(variant.alleles)}  "
        f"effect={row['effect_allele']} other={row['noneffect_allele']}  "
        f"n_matched={n_matched}"
    )
    if mismatches:
        print(f"      {len(mismatches)} samples disagree with the oracle:")
        for s, (exp, act) in list(mismatches.items())[:5]:
            print(f"        {s}: expected {exp}, library gave {act}")
    if n_imputed:
        print(
            f"      NOTE: {n_imputed} sample(s) were never called here; the "
            f"library imputed hom-ref for them (not counted as a failure)"
        )
    return ok

## Checks 2-7 -- the scoring scenarios

Each one mirrors a test in `tests/integration/test_allele_matching.py`, but
against a real variant and a real cohort.

**Check 5, multi-allelic with a REF effect allele**, is the one that matters
most. It is where Finding 6 lived: splitting downcodes every *other* ALT to the
reference, so a sample carrying a different ALT looked homozygous-reference,
collected the full `2w` offset, and was scored as carrying two copies of an
allele it holds one of -- or none of. The cell after this one counts how many
samples actually carry another ALT, because if none does, check 5 passes
trivially and proves nothing.

**Check 7, the no-call**, is the other end of the same distinction. A no-call
is an entry that *exists* but whose genotype is *missing* -- an **unknown**
genotype, not a homozygous-reference one. It must score 0, not `2w`. Getting
this wrong is what sank the old non-split scoring path, which handed every
no-call two copies of the reference.

The effect allele here is the REF base, so a no-call sample that wrongly
collected the offset would be scored as a homozygote for an allele nobody knows
it carries.

In [ ]:
SKIPPED = []

bi = FOUND["biallelic"][0]
ma = FOUND["multiallelic"][0]
nm = FOUND["non_minimal"][0]

check("2. biallelic, effect = ALT", bi, 1, ref_is_effect=False)
check("3. biallelic, effect = REF (the 2w offset)", bi, 1, ref_is_effect=True)
check("4. multi-allelic, effect = ALT", ma, 1, ref_is_effect=False)
check(
    "5. multi-allelic, effect = REF  <-- Finding 6", ma, 1, ref_is_effect=True
)
check("6. non-minimal representation (suffix trim)", nm, 1, ref_is_effect=False)

if FOUND["no_call"]:
    nc = FOUND["no_call"][0]
    check(
        "7. no-call, effect = REF (must not impute to 2 copies)",
        nc,
        1,
        ref_is_effect=True,
    )
else:
    # A skipped check is NOT a passed one. Record it so the summary says so --
    # a check that quietly disappears from the table is worse than a red one.
    SKIPPED.append("7. no-call, effect = REF -- no no-call entry in WINDOW")
    print("SKIP  7. no-call -- none found in this window")

### How meaningful was check 5?

Check 5 only bites if some sample actually carries an ALT that the weights row
does **not** name. If nobody does, the check passes trivially and proves
nothing. Count them.

In [ ]:
# Samples carrying allele index >= 2 at the multi-allelic site -- i.e. an ALT
# other than the one the weights row named.
key = f"{ma.locus}|{','.join(ma.alleles)}"
row_key = hl.str(dense.locus) + "|" + hl.delimit(dense.alleles, ",")
at_site = dense.filter_rows(row_key == key)

other = (
    at_site.select_cols(
        n_other=hl.agg.count_where(
            hl.is_defined(at_site.GT)
            & ((at_site.GT[0] >= 2) | (at_site.GT[1] >= 2))
        )
    )
    .cols()
    .to_pandas()
)

n_carriers = int((other.n_other > 0).sum())
print(f"{ma.locus} {list(ma.alleles)}")
print(f"samples carrying an ALT the weights row did NOT name: {n_carriers}")

if n_carriers == 0:
    print(
        "\nWARNING: no sample carries the other ALT here, so check 5 could "
        "not have failed even if Finding 6 were unfixed. Widen WINDOW or "
        "pick a multi-allelic site with more carriers to make it meaningful."
    )
else:
    print(
        f"\nOK: check 5 is meaningful. Before Finding 6 was fixed, these "
        f"{n_carriers} samples would each have been over-credited."
    )

## Check 8 -- an unmatched variant contributes nothing

A weights row naming a variant that is **not** in the callset must score 0 for
everyone -- and crucially must contribute **no offset**. Its effect allele here
is a REF base, so if the offset were credited for unmatched rows, every sample
in the cohort would be handed `2w` for a variant nobody carries.

In [ ]:
# Same locus as the real biallelic SNP, but naming an ALT that is not there.
absent_alt = next(
    b for b in "ACGT" if b not in bi.alleles and b != bi.alleles[0]
)
ghost = {
    "chr": bi.locus.contig,
    "pos": bi.locus.position,
    "effect_allele": bi.alleles[0],  # a REF base: would carry a 2w offset
    "noneffect_allele": absent_alt,
    "weight": 1.0,
}

prs, n_matched = library_score([ghost])
ok = n_matched == 0 and all(abs(v) < 1e-9 for v in prs.values())
RESULTS.append(("8. unmatched variant contributes nothing", ok, 0))

print(f"{'PASS' if ok else 'FAIL'}  8. unmatched variant contributes nothing")
print(
    f"      named {bi.locus} {bi.alleles[0]}/{absent_alt}, which is not in "
    f"the VDS"
)
print(f"      n_matched={n_matched}, distinct scores={set(prs.values())}")

## Check 9 -- chunking partitions the weights

Chunking is what makes the library affordable on the real VDS. It is only
correct if the chunks partition the variants exactly: an overlap double-counts
a variant, a gap drops one, and either way the score is still a clean float.

One variant per chunk must give the same answer as one chunk for all.

In [ ]:
all_rows = [
    weights_row(v, 1, ref_is_effect=(i % 2 == 0))
    for i, v in enumerate(SELECTED)
]


def totals(chunk_size):
    raw = hl.Table.parallelize(
        all_rows,
        hl.tstruct(
            chr=hl.tstr,
            pos=hl.tint32,
            effect_allele=hl.tstr,
            noneffect_allele=hl.tstr,
            weight=hl.tfloat64,
        ),
    )
    config = PRSConfig(chunk_size=chunk_size)
    weights, n_chunks = _prepare_weights_for_chunking(
        weights_table=raw, config=config, validate_table=True
    )
    parts = _process_chunks(
        full_weights_table=weights, n_chunks=n_chunks, vds=vds, config=config
    )
    combined = pd.concat(parts, ignore_index=True)
    summed = combined.groupby("person_id")["prs"].sum()
    return n_chunks, summed.to_dict()


n_one, one_chunk = totals(chunk_size=None)
n_many, per_variant = totals(chunk_size=1)

ok = all(abs(one_chunk[s] - per_variant[s]) < 1e-9 for s in one_chunk)
RESULTS.append(("9. chunking partitions the weights", ok, 0))

print(f"{'PASS' if ok else 'FAIL'}  9. chunking partitions the weights")
print(
    f"      {n_one} chunk vs {n_many} chunks, {len(one_chunk)} samples compared"
)

## Check 10 -- batch agrees with single

The batch calculator never filters rows; it masks with `hl.if_else` where the
single-score path uses `filter_rows`. Two different mechanisms that must
produce the same number -- including for the hom-ref offset, which batch has to
gate on `is_valid_{score}` explicitly.

In [ ]:
raw = hl.Table.parallelize(
    all_rows,
    hl.tstruct(
        chr=hl.tstr,
        pos=hl.tint32,
        effect_allele=hl.tstr,
        noneffect_allele=hl.tstr,
        weight=hl.tfloat64,
    ),
)
config = PRSConfig(include_n_matched=True)

single, _ = library_score(all_rows)

prepared, _ = _prepare_batch_weights_data({"s1": raw}, config)
df = _calculate_prs_chunk_batch(vds, {"s1": raw}, prepared, config).to_pandas()
batch = dict(zip(df["s"], df["s1"].astype(float), strict=True))

ok = all(abs(single[s] - batch[s]) < 1e-9 for s in single)
RESULTS.append(("10. batch agrees with single", ok, 0))

print(f"{'PASS' if ok else 'FAIL'}  10. batch agrees with single")
print(f"      {len(single)} samples compared")

## Summary

In [ ]:
summary = pd.DataFrame(RESULTS, columns=["check", "passed", "n_mismatched"])
print(summary.to_string(index=False))

n_failed = int((~summary.passed).sum())
print()

if SKIPPED:
    print("SKIPPED -- these scenarios were NOT validated on real data:")
    for name in SKIPPED:
        print(f"  {name}")
    print()

if n_failed:
    raise AssertionError(
        f"{n_failed} check(s) FAILED against the real All of Us VDS.\n"
        "This is a real finding: either a bug in aoutools, or a false "
        "assumption in tests/integration/. Do not adjust the oracle to make "
        "it pass -- the oracle is independent of the code under test, which "
        "is the whole point."
    )

print(f"All {len(summary)} checks passed against the real All of Us VDS.")
if SKIPPED:
    print(
        f"But {len(SKIPPED)} check(s) never ran. Widen WINDOW until they do: "
        "an un-run check is an unvalidated scenario, not a green one, and "
        "this notebook is what gates the release."
    )

## Runtime

In [ ]:
elapsed = time.perf_counter() - NOTEBOOK_START
print(f"Whole-notebook wall time: {elapsed:.1f} s  ({elapsed / 60:.1f} min)")
print(
    "Measured from the setup cell to here, so it is only meaningful on a clean "
    "top-to-bottom run. Each check scores a single variant, so most of this is "
    "fixed Spark/VDS overhead rather than the scoring arithmetic."
)